# M0 算法可行性预实测 (ADR-014)

目标：100 张样图 (`samples/{cat,face,pet,scene}/` 各 25)
- 端到端 P95 ≤ 10s
- 人工评分优良率 ≥ 60%

通过则把 `pipeline/` 各模块作为种子搬到 Phase 1 后端 `algo-api` 容器。

**先决条件**：
1. `data/mard_palette.json` 已就位（参考 `README.md §4.2`）
2. `samples/<cat|face|pet|scene>/` 各放 25 张真实照片
3. `uv sync` 已装好依赖

In [ ]:
import sys
from pathlib import Path

# 让 Notebook 能 import 上一级 pipeline/
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import csv
import json
import shutil
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from pipeline import run_pipeline

SAMPLES_DIR = PROJECT_ROOT / 'samples'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CATEGORIES = ['cat', 'face', 'pet', 'scene']
GRID = 48           # MVP normal 档
TARGET_COLORS = 24  # MVP normal 档

## 1. 预检查：样本与色卡是否就位

In [ ]:
palette_file = PROJECT_ROOT / 'data' / 'mard_palette.json'
assert palette_file.exists(), f'缺少色卡 {palette_file}, 见 README.md §4.2'
with open(palette_file) as f:
    palette = json.load(f)
print(f'Mard 色卡: {len(palette)} 个色号')

for cat in CATEGORIES:
    files = sorted((SAMPLES_DIR / cat).glob('*'))
    files = [f for f in files if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')]
    print(f'samples/{cat}: {len(files)} 张')
    if len(files) < 25:
        print(f'  ⚠️ 不足 25 张，先补图再跑后续 cell')

## 2. 批跑 100 张样图，记录每步耗时

In [ ]:
rows = []
process = psutil.Process()

for cat in CATEGORIES:
    files = sorted((SAMPLES_DIR / cat).glob('*'))
    files = [f for f in files if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')][:25]
    for idx, fp in enumerate(files):
        sample_id = f'{cat}_{idx:02d}'
        out_dir = RESULTS_DIR / sample_id
        out_dir.mkdir(exist_ok=True)

        rss_before = process.memory_info().rss / 1024 / 1024
        try:
            r = run_pipeline(fp, grid=GRID, target_colors=TARGET_COLORS)
        except Exception as e:
            print(f'❌ {sample_id} failed: {e}')
            rows.append({'sample_id': sample_id, 'category': cat, 'grid': GRID,
                          'target_colors': TARGET_COLORS, 'total_seconds': 9999,
                          'score_1_to_5': '', 'notes': f'ERROR: {e}'})
            continue
        rss_after = process.memory_info().rss / 1024 / 1024

        # 保存原图缩略 + 索引矩阵 + 简单预览
        shutil.copy(fp, out_dir / f'src{fp.suffix}')
        np.save(out_dir / 'index_grid.npy', r.index_grid)
        with open(out_dir / 'color_summary.json', 'w', encoding='utf-8') as f:
            json.dump(r.color_summary, f, ensure_ascii=False, indent=2)

        # 渲染快速预览（每色用调色板 RGB 画块）
        preview = np.full((GRID, GRID, 3), 255, dtype=np.uint8)
        for y in range(GRID):
            for x in range(GRID):
                pi = int(r.index_grid[y, x])
                if pi >= 0:
                    e = palette[pi]
                    preview[y, x] = (e['r'], e['g'], e['b'])
        Image.fromarray(preview).resize((480, 480), Image.NEAREST).save(out_dir / 'preview.png')

        rows.append({
            'sample_id': sample_id,
            'category': cat,
            'grid': GRID,
            'target_colors': TARGET_COLORS,
            'total_seconds': round(r.total_seconds, 3),
            'rss_delta_mb': round(rss_after - rss_before, 1),
            'colors': r.color_summary['color_count'],
            'beads': r.color_summary['foreground_cells'],
            **{f'step_{t.name}_ms': round(t.duration_ms, 1) for t in r.timings},
            'score_1_to_5': '',
            'notes': '',
        })
        print(f'✅ {sample_id}  total={r.total_seconds:.2f}s  colors={r.color_summary["color_count"]}  beads={r.color_summary["foreground_cells"]}')

df = pd.DataFrame(rows)
df.to_csv(RESULTS_DIR / 'timing_report.csv', index=False)
df

## 3. 性能汇总：P95 / 各步耗时分布

In [ ]:
print(f'总样本: {len(df)}')
print(f'P50 总耗时: {df.total_seconds.quantile(0.5):.2f} s')
print(f'P95 总耗时: {df.total_seconds.quantile(0.95):.2f} s   (目标 ≤ 10.0 s)')
print(f'P99 总耗时: {df.total_seconds.quantile(0.99):.2f} s')
print('\n各步骤耗时 P95 (ms):')
step_cols = [c for c in df.columns if c.startswith('step_')]
for c in step_cols:
    print(f'  {c}: {df[c].quantile(0.95):.0f} ms')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.total_seconds.hist(bins=20, ax=ax[0])
ax[0].axvline(10, color='r', linestyle='--', label='P95 target 10s')
ax[0].set_xlabel('total seconds'); ax[0].set_title('End-to-end timing'); ax[0].legend()
df[step_cols].mean().plot.bar(ax=ax[1])
ax[1].set_ylabel('mean ms'); ax[1].set_title('Per-step mean cost')
plt.tight_layout(); plt.show()

## 4. 导出评分模板

`scoring/score_pending.csv` 会被这一步覆盖为待评分行（模板 `score_template.csv` 不动）。
请逐张打开 `data/results/<sample_id>/preview.png` 给 1~5 分，填到该 CSV 的 `score_1_to_5` 列。

完成后跑：`uv run python scoring/score_summary.py scoring/score_pending.csv`

In [ ]:
scoring_csv = PROJECT_ROOT / 'scoring' / 'score_pending.csv'
header = ['sample_id', 'category', 'grid', 'target_colors', 'total_seconds', 'score_1_to_5', 'notes']
with open(scoring_csv, 'w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=header)
    w.writeheader()
    for _, row in df.iterrows():
        w.writerow({k: row.get(k, '') for k in header})
print(f'已写出 {scoring_csv}，共 {len(df)} 行待打分')

## 5. 阻塞门判定

- ✅ PASS：`docs/07-algo-spec.md §6.1` 阈值全部达标 → 准备 Phase 1 W1
- ❌ FAIL：见 `README.md §5` 失败路径